# A Hybrid Deep Reinforcement Learning and LLM Framework for Real-Time Distillation Optimization
#### AI Agents for XAI and process improvement in CDU + NSU + VDU

This notebook builds an AI Agent system using **Google Gemini** models for intelligent analysis of the
three-column distillation system: **Atmospheric Distillation Unit (ADU)**, **Naphtha Stabilizer Unit (NSU)**,
and **Vacuum Distillation Unit (VDU)** — producing **13 product streams** optimized by a 16-dim RL agent.

## Agent Personas

| Persona | Role | Focus Areas |
|---------|------|-------------|
| **Process Engineer** | Full 3-column system analysis | Column performance, 13-product yields, D95% specs, profit optimization |
| **Daily Report Developer** | Automated report generation | Daily summaries, KPIs, cross-column yield trends |
| **Corrosion Expert** | ADU overhead corrosion analysis | Dew point corrosion, NH₄Cl salt deposition, chemical treatment |

---


## 1. Setup & Configuration

In [130]:
import os
import json
import sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional
import time as _time
import requests as _requests

from google import genai
from google.genai import types
import pandas as pd
import numpy as np


# Add project root to path
PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: d:\github\Distillation-column-agent


### Configuration

In [152]:


# ── Primary: Google Gemini ────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
print(f"GEMINI_API_KEY : {'set' if GEMINI_API_KEY else 'not set'}")

if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Primary LLM    : Gemini (google-genai SDK)")
else:
    client = None
    print("  → Set GEMINI_API_KEY env var, or uncomment the line above to paste it directly.")

# ── Fallback: Nvidia Nemotron via OpenRouter ──────────────────────────────
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
# Uncomment and paste key here if the env var is not set:
OPENROUTER_MODEL   = "nvidia/nemotron-3-nano-30b-a3b:free"
OPENROUTER_BASE    = "https://openrouter.ai/api/v1/chat/completions"

print(f"OPENROUTER_KEY : {'set' if OPENROUTER_API_KEY else 'not set'}")

if not GEMINI_API_KEY and not OPENROUTER_API_KEY:
    print("\n No LLM API key configured — agents will run in offline/demo mode.")
elif not GEMINI_API_KEY and OPENROUTER_API_KEY:
    print(f"Fallback LLM   : {OPENROUTER_MODEL}")

# ── Model tiers for cost control ────────────────────────
MODEL_TIERS = {
    "lite":     "gemini-2.0-flash-lite",
    "standard": "gemini-2.0-flash",
    "premium":  "gemini-2.5-flash",
}


def _openrouter_chat(system_prompt: str, user_message: str,
                     model: str = OPENROUTER_MODEL) -> str:
    """Call OpenRouter (OpenAI-compatible REST) and return response text."""
    if not OPENROUTER_API_KEY:
        raise ValueError(
            "OPENROUTER_API_KEY is not set. "
            "Set the env var or uncomment the OPENROUTER_API_KEY line in the config cell."
        )
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/Distillation-column-agent",
        "X-Title": "CDU Optimizer AI Agent",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        "stream": False,
    }
    resp = _requests.post(OPENROUTER_BASE, headers=headers, json=payload, timeout=90)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


GEMINI_API_KEY : set
Primary LLM    : Gemini (google-genai SDK)
OPENROUTER_KEY : set


## 2. Data Infrastructure


**Live DWSIM grounding** with TTL-cached reads and mock fallback

**Deterministic numeric module** — mass balance, profit, D95 checks are computed in code, not by LLM

**Compact context builder** — sends only key KPIs to agents, reducing token bloat

**Audit logger** — every agent call is logged to `Report/agent_audit.jsonl` for reproducibility

### 2a. Prices & Training Metrics

In [153]:
def load_prices(scenario="default"):
    """Load product prices from the data file."""
    prices_path = PROJECT_ROOT / "backend" / "data" / "prices.json"
    if prices_path.exists():
        with open(prices_path) as f:
            data = json.load(f)
        key = f"prices_{scenario}"
        if key in data:
            return data[key].get("prices", {})
    # Fallback
    return {
        "Uncondensed_Gas": 0.30, "Heavy_Naphtha": 0.60, "SKO": 0.75,
        "Light_Gas_Oil": 0.70, "Heavy_Gas_Oil": 0.70,
        "StabOffGas": 0.30, "LPG": 0.65, "SRN": 0.75,
        "Offgas": 0.30, "Vacuum_Diesel": 0.70, "Vacuum_Gas_Oil": 0.50,
        "Hotwell_Oil": 0.50, "Vac_residue": 0.35,
        "Feed_Crude": 0.40,
    }


def load_latest_training_metrics():
    """Load the latest training checkpoint metrics."""
    cp_dir = PROJECT_ROOT / "checkpoints"
    metrics_files = sorted(cp_dir.glob("*_metrics.json"), reverse=True)
    if metrics_files:
        with open(metrics_files[0]) as f:
            return json.load(f)
    return None


### 2b. DWSIM Bridge with TTL Cache + Mock Fallback

In [159]:
FLOWSHEET_PATH = str(PROJECT_ROOT / "Sim_models" / "main_sim.dwxmz")

# D95% spec limits (°C) — For using in sanity checks and prompts
## This can be stored in json files or a database in a more complex implementation
## For simplicity, we hardcode it here for now
D95_SPECS = {
    "Heavy_Naphtha": 220.0, "SKO": 300.0, "Light_Gas_Oil": 370.0,
    "Heavy_Gas_Oil": 385.0, "Vacuum_Diesel": 385.0, "Vacuum_Gas_Oil": 520.0,
}

# Product keys grouped by column
ADU_PRODUCTS = ["Uncondensed_Gas", "Heavy_Naphtha", "SKO", "Light_Gas_Oil", "Heavy_Gas_Oil"]
NSU_PRODUCTS = ["StabOffGas", "LPG", "SRN"]
VDU_PRODUCTS = ["Offgas", "Vacuum_Diesel", "Vacuum_Gas_Oil", "Hotwell_Oil", "Vac_residue"]
ALL_PRODUCTS = ADU_PRODUCTS + NSU_PRODUCTS + VDU_PRODUCTS

# ── TTL cache for bridge reads ────────────────────────────────────────────
_bridge_cache: dict = {"state": None, "timestamp": 0.0, "ttl": 30.0}  # 30 s TTL


def _get_mock_column_state() -> dict:
    """Representative operating snapshot (fallback when DWSIM unavailable)."""
    return {
        "flow_Uncondensed_Gas": 25.0, "flow_Heavy_Naphtha": 106.0,
        "flow_SKO": 43.0, "flow_Light_Gas_Oil": 51.0, "flow_Heavy_Gas_Oil": 69.0,
        "flow_StabOffGas": 18.0, "flow_LPG": 32.0, "flow_SRN": 56.0,
        "flow_Offgas": 12.0, "flow_Vacuum_Diesel": 38.0,
        "flow_Vacuum_Gas_Oil": 72.0, "flow_Hotwell_Oil": 14.0, "flow_Vac_residue": 95.0,
        "temp_Uncondensed_Gas": 50.0, "temp_Heavy_Naphtha": 155.0,
        "temp_SKO": 220.0, "temp_Light_Gas_Oil": 280.0, "temp_Heavy_Gas_Oil": 340.0,
        "temp_StabOffGas": 45.0, "temp_LPG": 50.0, "temp_SRN": 90.0,
        "temp_Offgas": 65.0, "temp_Vacuum_Diesel": 250.0,
        "temp_Vacuum_Gas_Oil": 350.0, "temp_Hotwell_Oil": 380.0, "temp_Vac_residue": 410.0,
        "d95_Heavy_Naphtha": 195.0, "d95_SKO": 275.0, "d95_Light_Gas_Oil": 355.0,
        "d95_Heavy_Gas_Oil": 375.0, "d95_Vacuum_Diesel": 370.0, "d95_Vacuum_Gas_Oil": 500.0,
        "top_temperature": 50.0, "bottom_temperature": 340.0,
        "feed_temperature": 365.0, "feed_flow_rate": 4736.0,
        "top_pressure": 101.0, "bottom_pressure": 116.0,
        "condenser_duty": 42000.0, "reboiler_duty": 46000.0,
        "heater_duty": 8500.0, "reflux_ratio": 5.0,
        "nsu_top_temperature": 45.0, "nsu_bottom_temperature": 155.0,
        "nsu_top_pressure": 800.0, "nsu_reflux_ratio": 4.0,
        "nsu_condenser_duty": 8500.0, "nsu_reboiler_duty": 9200.0,
        "vac_top_pressure": 8.0, "vac_bottom_pressure": 15.0,
        "vac_condenser_duty": 12000.0, "vac_reboiler_duty": 18000.0,
        "vac_bottom_temperature": 410.0, "vac_furnace_duty": 4200.0, "vac_reflux_ratio": 3.5,
        "overhead_temperature": 50.0, "overhead_pressure": 101.0,
        "overhead_water_content": 0.02, "overhead_hcl_ppm": 5.0,
        "overhead_h2s_ppm": 15.0, "overhead_nh3_ppm": 8.0,
    }


def get_column_state(flowsheet_path: str | None = None, use_cache: bool = True) -> dict:
    """
    Fetch live column state from DWSIM (primary) or return mock (fallback).

    - Uses a TTL cache (default 30 s) to avoid re-opening the COM bridge.
    - Supplements bridge output with operating-point data and overhead-corrosion estimates.
    """
    # Check cache first
    if use_cache and _bridge_cache["state"] is not None:
        age = _time.time() - _bridge_cache["timestamp"]
        if age < _bridge_cache["ttl"]:
            print(f"📦 Using cached DWSIM state ({age:.0f}s old, TTL={_bridge_cache['ttl']:.0f}s)")
            return _bridge_cache["state"]

    if flowsheet_path:
        bridge = None
        try:
            from backend.core.dwsim_bridge import DWSIMBridge
            bridge = DWSIMBridge(flowsheet_path)
            bridge.load()

            state = bridge.get_column_state()

            # Augment with operating-point data (NSU reflux, VDU reflux, etc.)
            try:
                op = bridge.get_current_operating_point()
                state.setdefault("reflux_ratio",     op.get("reflux_ratio", 5.0))
                state.setdefault("nsu_reflux_ratio", op.get("nsu_reflux_ratio", 4.0))
                state.setdefault("nsu_reboiler_temp", op.get("nsu_reboiler_temp", 155.0))
                state.setdefault("vac_reflux_ratio", op.get("vac_reflux_ratio", 3.5))
            except Exception:
                state.setdefault("reflux_ratio", 5.0)
                state.setdefault("nsu_reflux_ratio", 4.0)
                state.setdefault("vac_reflux_ratio", 3.5)

            # Fill fields not modelled in DWSIM from mock estimates
            mock = _get_mock_column_state()
            for key in ("heater_duty", "vac_furnace_duty",
                        "nsu_top_temperature", "nsu_bottom_temperature",
                        "nsu_top_pressure", "nsu_condenser_duty", "nsu_reboiler_duty",
                        "overhead_temperature", "overhead_pressure",
                        "overhead_water_content", "overhead_hcl_ppm",
                        "overhead_h2s_ppm", "overhead_nh3_ppm"):
                state.setdefault(key, mock[key])

            # Use live top-of-column values for overhead corrosion inputs
            state["overhead_temperature"] = state.get("top_temperature", state["overhead_temperature"])
            state["overhead_pressure"]    = state.get("top_pressure", state["overhead_pressure"])

            # Update cache
            _bridge_cache["state"] = state
            _bridge_cache["timestamp"] = _time.time()
            print("DWSIM live data loaded successfully.")
            return state

        except Exception as e:
            print(f"DWSIM connection failed: {e}")
            print("   → Falling back to representative mock data.")
        finally:
            if bridge is not None:
                try:
                    bridge.close()
                except Exception:
                    pass

    print("ℹUsing representative mock column state (DWSIM not connected).")
    return _get_mock_column_state()

### 2c. Deterministic Numeric Module

Compute mass balance, profit, D95 compliance IN CODE

In [155]:
def compute_mass_balance(state: dict) -> dict:
    """Deterministic mass-balance check for all three columns."""
    feed = state.get("feed_flow_rate", 0.0)
    total_product = sum(state.get(f"flow_{p}", 0.0) for p in ALL_PRODUCTS)
    gap = feed - total_product
    gap_pct = (gap / feed * 100) if feed > 0 else 0.0
    return {
        "feed_flow_kg_h": round(feed, 1),
        "total_product_kg_h": round(total_product, 1),
        "gap_kg_h": round(gap, 1),
        "gap_pct": round(gap_pct, 2),
        "closure_ok": abs(gap_pct) < 2.0,
    }


def compute_profit(state: dict, prices: dict) -> dict:
    """Deterministic hourly profit calculation."""
    revenue_by_product = {}
    for p in ALL_PRODUCTS:
        flow = state.get(f"flow_{p}", 0.0)
        price = prices.get(p, 0.0)
        revenue_by_product[p] = round(flow * price, 2)
    total_revenue = sum(revenue_by_product.values())
    feed_cost = state.get("feed_flow_rate", 0.0) * prices.get("Feed_Crude", 0.40)
    net_profit = total_revenue - feed_cost
    return {
        "revenue_by_product": revenue_by_product,
        "total_revenue_hr": round(total_revenue, 2),
        "feed_cost_hr": round(feed_cost, 2),
        "net_profit_hr": round(net_profit, 2),
        "top_3_contributors": sorted(revenue_by_product.items(), key=lambda x: x[1], reverse=True)[:3],
    }


def compute_d95_compliance(state: dict) -> dict:
    """Check D95% temperature against spec limits."""
    results = {}
    for product, limit in D95_SPECS.items():
        actual = state.get(f"d95_{product}", None)
        results[product] = {
            "d95_actual_C": actual,
            "d95_limit_C": limit,
            "pass": actual is not None and actual <= limit,
            "margin_C": round(limit - actual, 1) if actual is not None else None,
        }
    return results


def run_sanity_checks(state: dict, prices: dict) -> dict:
    """
    Run all deterministic sanity checks on column state.
    Returns a dict with pass/fail flags and details.
    """
    mb = compute_mass_balance(state)
    d95 = compute_d95_compliance(state)
    profit = compute_profit(state, prices)

    # Non-negative flow check
    negative_flows = {p: state.get(f"flow_{p}", 0.0) for p in ALL_PRODUCTS
                      if state.get(f"flow_{p}", 0.0) < 0}

    # Pressure bounds
    pressure_ok = (
        90 <= state.get("top_pressure", 0) <= 120
        and 5 <= state.get("vac_top_pressure", 0) <= 20
    )

    all_ok = (
        mb["closure_ok"]
        and all(v["pass"] for v in d95.values() if v["d95_actual_C"] is not None)
        and len(negative_flows) == 0
        and pressure_ok
    )

    return {
        "all_passed": all_ok,
        "mass_balance": mb,
        "d95_compliance": d95,
        "profit": profit,
        "negative_flows": negative_flows,
        "pressure_bounds_ok": pressure_ok,
    }

### 2d. Compact Context Builder

Sends only a concise KPI summary to agents, not the full 50+ key dict.

In [160]:
def build_compact_context(state: dict, prices: dict, training_data: dict | None = None) -> dict:
    """Build a token-efficient context dict for agent prompts."""
    checks = run_sanity_checks(state, prices)
    mb = checks["mass_balance"]
    profit = checks["profit"]
    d95 = checks["d95_compliance"]

    # Product summary table: flow, temp, revenue per product
    products = {}
    for p in ALL_PRODUCTS:
        products[p] = {
            "flow_kg_h": state.get(f"flow_{p}", 0.0),
            "temp_C": state.get(f"temp_{p}", 0.0),
            "revenue_hr": profit["revenue_by_product"].get(p, 0.0),
        }
        if p in D95_SPECS:
            products[p]["d95_C"] = state.get(f"d95_{p}")
            products[p]["d95_limit_C"] = D95_SPECS[p]
            products[p]["d95_pass"] = d95[p]["pass"]

    ctx = {
        "kpi_summary": {
            "feed_kg_h": mb["feed_flow_kg_h"],
            "total_product_kg_h": mb["total_product_kg_h"],
            "mass_balance_gap_pct": mb["gap_pct"],
            "mass_balance_ok": mb["closure_ok"],
            "net_profit_hr": profit["net_profit_hr"],
            "total_revenue_hr": profit["total_revenue_hr"],
            "feed_cost_hr": profit["feed_cost_hr"],
            "d95_all_pass": all(v["pass"] for v in d95.values() if v["d95_actual_C"] is not None),
            "pressure_bounds_ok": checks["pressure_bounds_ok"],
        },
        "products": products,
        "column_conditions": {
            "ADU": {
                "top_temp_C": state.get("top_temperature"),
                "bottom_temp_C": state.get("bottom_temperature"),
                "reflux_ratio": state.get("reflux_ratio"),
                "top_pressure_kPa": state.get("top_pressure"),
                "dp_kPa": round(state.get("bottom_pressure", 0) - state.get("top_pressure", 0), 1),
                "condenser_duty_kW": state.get("condenser_duty"),
                "reboiler_duty_kW": state.get("reboiler_duty"),
            },
            "NSU": {
                "top_temp_C": state.get("nsu_top_temperature"),
                "bottom_temp_C": state.get("nsu_bottom_temperature"),
                "reflux_ratio": state.get("nsu_reflux_ratio"),
                "top_pressure_kPa": state.get("nsu_top_pressure"),
            },
            "VDU": {
                "top_pressure_kPa": state.get("vac_top_pressure"),
                "bottom_temp_C": state.get("vac_bottom_temperature"),
                "reflux_ratio": state.get("vac_reflux_ratio"),
                "dp_kPa": round(state.get("vac_bottom_pressure", 0) - state.get("vac_top_pressure", 0), 1),
            },
        },
        "overhead_corrosion": {
            "temp_C": state.get("overhead_temperature"),
            "pressure_kPa": state.get("overhead_pressure"),
            "water_frac": state.get("overhead_water_content"),
            "hcl_ppm": state.get("overhead_hcl_ppm"),
            "h2s_ppm": state.get("overhead_h2s_ppm"),
            "nh3_ppm": state.get("overhead_nh3_ppm"),
        },
    }

    if training_data:
        fm = training_data.get("final_metrics", {})
        ctx["rl_agent"] = {
            "algorithm": training_data.get("config", {}).get("algorithm", "SAC"),
            "episodes": fm.get("episode", 0),
            "best_reward": fm.get("best_reward", 0),
            "avg_reward": fm.get("avg_reward", 0),
        }

    return ctx

### 2e. Audit Logger

In [161]:
AUDIT_LOG_PATH = PROJECT_ROOT / "Report" / "agent_audit.jsonl"


def audit_log(agent_name: str, question: str, context_keys: list,
              response_text: str, validation: dict | None = None):
    """Append one line to the JSONL audit log."""
    AUDIT_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    entry = {
        "timestamp": datetime.now().isoformat(),
        "agent": agent_name,
        "question": question[:500],
        "context_keys": context_keys,
        "response_length": len(response_text),
        "response_preview": response_text[:300],
        "validation": validation,
    }
    with open(AUDIT_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, default=str) + "\n")

### 2f. Loading all data

In [162]:
prices        = load_prices()
training_data = load_latest_training_metrics()
column_state  = get_column_state(flowsheet_path=FLOWSHEET_PATH)

# Run deterministic checks immediately
sanity = run_sanity_checks(column_state, prices)

print(f"\nPrices loaded : {len([k for k in prices if k != 'Feed_Crude'])} products + feed cost")
print(f"Training data : {'available' if training_data else 'none'}")
print(f"Column state  : {len(column_state)} keys")
print(f"\n{'='*60}")
print(f"  Mass balance   : {'Ok' if sanity['mass_balance']['closure_ok'] else 'X'}"
      f"  gap={sanity['mass_balance']['gap_pct']:.1f}%")
print(f"  D95 compliance : {'Ok' if all(v['pass'] for v in sanity['d95_compliance'].values() if v['d95_actual_C'] is not None) else 'X'}")
print(f"  Pressure bounds: {'Ok' if sanity['pressure_bounds_ok'] else 'X'}")
print(f"  Net profit     : ${sanity['profit']['net_profit_hr']:.2f}/hr")
print(f"{'='*60}")


2026-03-25 21:48:23.989 | INFO     | backend.core.dwsim_bridge:__init__:230 - DWSIMBridge created for d:\github\Distillation-column-agent\Sim_models\main_sim.dwxmz
2026-03-25 21:48:25.304 | INFO     | backend.core.dwsim_bridge:load:238 - Flowsheet loaded
2026-03-25 21:48:25.334 | INFO     | backend.core.dwsim_bridge:get_current_operating_point:613 - Current operating point read from simulation: {'reflux_ratio': 5.0, 'hn_draw_temp': 242.05464319653697, 'sko_draw_temp': 244.30785716119203, 'ld_draw_temp': 246.80129542263, 'hd_draw_temp': 247.95657528720608, 'atmos_reboiler_temp': 365.0, 'nsu_reflux_ratio': 5.0, 'nsu_reboiler_temp': 155.0, 'vac_reflux_ratio': 5.0, 'vac_reboiler_temp': 360.0, 'vac_diesel_draw_temp': 244.11702854621706, 'vgo_draw_temp': 290.738758705102, 'atmos_top_pressure': 170.0, 'atmos_dp': 40.0, 'vac_top_pressure': 20.0, 'vac_dp': 50.0}
2026-03-25 21:48:25.335 | INFO     | backend.core.dwsim_bridge:close:276 - DWSIM resources released


DWSIM live data loaded successfully.

Prices loaded : 10 products + feed cost
Training data : available
Column state  : 69 keys

  Mass balance   : X  gap=-4.7%
  D95 compliance : X
  Pressure bounds: X
  Net profit     : $-216734.40/hr


---

## 3. Agent Base Class

A reusable base for all AI agent personas with:
- **Structured JSON output** mode with schema validation (improvement 2)
- **Sanity checks & re-prompting** loop — automatically retries if response fails validation (improvement 4)
- **Compact context** — uses `build_compact_context()` instead of raw state dicts (improvement 5)
- **Audit logging** — every query/response is recorded to JSONL (improvement 6)
- **Model-tier selection** — `lite` / `standard` / `premium` for cost control (improvement 8)


In [ ]:
# ── JSON schema for structured agent responses (improvement 2) ────────────
AGENT_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "summary":          {"type": "string", "description": "1-2 sentence executive summary"},
        "analysis":         {"type": "string", "description": "Detailed technical analysis"},
        "recommendations":  {"type": "array", "items": {"type": "string"}, "maxItems": 10},
        "risk_level":       {"type": "string", "enum": ["GREEN", "YELLOW", "RED"]},
        "confidence":       {"type": "number", "minimum": 0, "maximum": 1},
        "numeric_results":  {"type": "object", "description": "Any numeric outputs"},
    },
    "required": ["summary", "analysis"],
}


def _validate_against_schema(data: dict, schema: dict) -> tuple[bool, list[str]]:
    """Lightweight JSON schema validation (no external deps)."""
    errors = []
    props = schema.get("properties", {})
    required = schema.get("required", [])

    for key in required:
        if key not in data:
            errors.append(f"Missing required field: '{key}'")

    for key, value in data.items():
        if key in props:
            expected_type = props[key].get("type")
            type_map = {"string": str, "number": (int, float), "array": list,
                        "object": dict, "boolean": bool}
            if expected_type and expected_type in type_map:
                if not isinstance(value, type_map[expected_type]):
                    errors.append(f"Field '{key}' expected {expected_type}, got {type(value).__name__}")
            if expected_type == "string" and "enum" in props[key]:
                if value not in props[key]["enum"]:
                    errors.append(f"Field '{key}' must be one of {props[key]['enum']}, got '{value}'")
            if expected_type == "number":
                if "minimum" in props[key] and value < props[key]["minimum"]:
                    errors.append(f"Field '{key}' below minimum {props[key]['minimum']}")
                if "maximum" in props[key] and value > props[key]["maximum"]:
                    errors.append(f"Field '{key}' above maximum {props[key]['maximum']}")

    return (len(errors) == 0, errors)


class BaseAgent:
    """
    Base AI Agent with Gemini (primary) and Nvidia Nemotron via OpenRouter (fallback).

    Improvements over v1:
    - ask()            → free-text response with audit logging
    - ask_structured() → JSON response with schema validation + re-prompting
    - Model-tier selection: lite / standard / premium
    - Compact context: uses build_compact_context() instead of raw state
    """

    def __init__(self, name: str, system_prompt: str,
                 model_tier: str = "standard"):
        self.name = name
        self.system_prompt = system_prompt
        self.model_tier = model_tier
        self.model_name = MODEL_TIERS.get(model_tier, MODEL_TIERS["standard"])
        self.conversation_history: list[dict] = []

        # Set up Gemini chat session if key is available
        if client:
            self._config = types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.3,
                max_output_tokens=8192,
            )
            self.chat = client.chats.create(model=self.model_name, config=self._config)
        else:
            self._config = None
            self.chat = None

    def _build_prompt(self, question: str, context: dict | None) -> str:
        """Build the full prompt from question + context."""
        ctx_parts = []
        if context:
            for key, value in context.items():
                if isinstance(value, dict):
                    ctx_parts.append(f"**{key}:**\n```json\n{json.dumps(value, indent=2, default=str)}\n```")
                elif isinstance(value, list):
                    ctx_parts.append(f"**{key}:**\n```json\n{json.dumps(value, default=str)}\n```")
                else:
                    ctx_parts.append(f"**{key}:** {value}")

        context_str = "\n\n".join(ctx_parts) if ctx_parts else "No additional context provided."
        return (
            f"**System Context:**\n{context_str}\n\n"
            f"**Question:** {question}\n\n"
            "Think step-by-step through the relevant principles, then provide your answer."
        )

    def _call_llm(self, prompt: str) -> str | None:
        """Try Gemini → OpenRouter → None."""
        answer = None

        # 1. Try Gemini
        if self.chat:
            try:
                response = self.chat.send_message(prompt)
                answer = response.text
            except Exception as e:
                print(f"  [Gemini error for {self.name}: {e}]")
                print("  → Falling back to OpenRouter / Nemotron …")

        # 2. Try OpenRouter fallback
        if answer is None and OPENROUTER_API_KEY:
            try:
                answer = _openrouter_chat(
                    system_prompt=self.system_prompt,
                    user_message=prompt,
                )
            except Exception as e:
                print(f"  [OpenRouter error for {self.name}: {e}]")

        return answer

    def ask(self, question: str, context: dict | None = None) -> str:
        """
        Send a question to the agent and get a free-text response.
        Logs to JSONL audit trail.
        """
        prompt = self._build_prompt(question, context)
        answer = self._call_llm(prompt)

        if answer is None:
            answer = self._offline_response(question)

        self.conversation_history.append({"role": "user", "content": question})
        self.conversation_history.append({"role": "assistant", "content": answer})

        # Audit log (improvement 6)
        audit_log(
            agent_name=self.name,
            question=question,
            context_keys=list(context.keys()) if context else [],
            response_text=answer,
        )

        return answer

    def ask_structured(self, question: str, context: dict | None = None,
                       schema: dict | None = None, max_retries: int = 2) -> dict:
        """
        Send a question and parse a structured JSON response.

        - Instructs the LLM to return valid JSON matching the schema.
        - Validates against `schema` (default: AGENT_RESPONSE_SCHEMA).
        - Re-prompts up to `max_retries` times on validation failure.
        """
        schema = schema or AGENT_RESPONSE_SCHEMA
        schema_hint = json.dumps({k: v.get("type", "any") for k, v in schema.get("properties", {}).items()})

        structured_instruction = (
            "IMPORTANT: Return your response as a single valid JSON object with these fields: "
            f"{schema_hint}. "
            f"Required fields: {schema.get('required', [])}. "
            "Do NOT include any text outside the JSON object. No markdown, no code fences."
        )

        prompt = self._build_prompt(question, context) + f"\n\n{structured_instruction}"

        for attempt in range(1 + max_retries):
            raw = self._call_llm(prompt)

            if raw is None:
                return {"summary": f"[{self.name} — Offline Mode]", "analysis": "No LLM available."}

            # Try to extract JSON from the response
            try:
                # Strip markdown code fences if present
                cleaned = raw.strip()
                if cleaned.startswith("```"):
                    cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned[3:]
                    if cleaned.endswith("```"):
                        cleaned = cleaned[:-3]
                    cleaned = cleaned.strip()
                parsed = json.loads(cleaned)
            except json.JSONDecodeError:
                if attempt < max_retries:
                    prompt = (
                        f"Your previous response was not valid JSON. Parse error occurred.\n"
                        f"Original question: {question}\n\n{structured_instruction}"
                    )
                    print(f" [{self.name}] JSON parse failed (attempt {attempt+1}), re-prompting…")
                    continue
                return {"summary": f"[{self.name}] Failed to produce valid JSON after {1+max_retries} attempts.",
                        "analysis": raw[:1000], "_raw": raw}

            # Validate against schema
            valid, errors = _validate_against_schema(parsed, schema)
            if valid:
                # Audit log with validation result
                audit_log(
                    agent_name=self.name,
                    question=question,
                    context_keys=list(context.keys()) if context else [],
                    response_text=json.dumps(parsed, default=str),
                    validation={"valid": True, "attempt": attempt + 1},
                )
                return parsed

            if attempt < max_retries:
                prompt = (
                    f"Your JSON response had validation errors: {errors}\n"
                    f"Please fix and return valid JSON.\n"
                    f"Original question: {question}\n\n{structured_instruction}"
                )
                print(f" [{self.name}] Schema validation failed (attempt {attempt+1}): {errors}")
                continue

            # Last attempt — return what we have with errors noted
            audit_log(
                agent_name=self.name,
                question=question,
                context_keys=list(context.keys()) if context else [],
                response_text=json.dumps(parsed, default=str),
                validation={"valid": False, "errors": errors, "attempt": attempt + 1},
            )
            parsed["_validation_errors"] = errors
            return parsed

    def _offline_response(self, question: str) -> str:
        """Fallback response when all APIs are unavailable."""
        return (
            f"[{self.name} — Offline Mode]\n\n"
            f"Question received: {question}\n\n"
            "Neither Gemini nor OpenRouter API is available.\n"
            "Set GEMINI_API_KEY or OPENROUTER_API_KEY to enable AI analysis."
        )

    def clear_history(self):
        """Reset conversation history and start a fresh chat session."""
        self.conversation_history = []
        if client and self._config:
            self.chat = client.chats.create(model=self.model_name, config=self._config)

    def __repr__(self):
        backend = "Gemini" if client else ("OpenRouter/Nemotron" if OPENROUTER_API_KEY else "Offline")
        return f"<{self.name} Agent | tier={self.model_tier} | backend={backend} | history={len(self.conversation_history)} msgs>"


print("BaseAgent class defined (with structured output, schema validation, re-prompting, audit logging)")
print(f"   Active backend: {'Gemini' if client else ('OpenRouter/Nemotron' if OPENROUTER_API_KEY else 'Offline')}")
print(f"   Model tiers: {MODEL_TIERS}")


✅ BaseAgent class defined (with structured output, schema validation, re-prompting, audit logging)
   Active backend: Gemini
   Model tiers: {'lite': 'gemini-2.0-flash-lite', 'standard': 'gemini-2.0-flash', 'premium': 'gemini-2.5-flash'}


---

## 4. Persona 1: Process Engineer

A process engineer who can analyze the full ADU + VDU system, evaluate column performance, product yields, and recommend operational improvements.

In [134]:
PROCESS_ENGINEER_PROMPT = """You are a Senior Process Engineer specializing in crude oil refining,
specifically Atmospheric Distillation Units (ADU), Naphtha Stabilizer Units (NSU), and Vacuum Distillation Units (VDU).

**Your expertise includes:**
- Petroleum refining process design and optimization
- TBP (True Boiling Point) and ASTM distillation curve analysis
- Mass and energy balance across three-column distillation systems
- Product yield optimization across 13 product streams
- D95% distillation temperature specifications and quality control
- Heat exchanger network optimization and furnace efficiency
- Process simulation using DWSIM (Peng-Robinson EOS for petroleum systems)
- RL-based optimization with 16-dimensional action space

**System Configuration (3-column unit):**
- Feed: Crude oil at ~365°C, ~4736 kg/h

- **ADU (Atmospheric Distillation Unit)** — 5 products:
    1. Uncondensed_Gas   (fuel gas,   ~0.30 $/kg, overhead)
    2. Heavy_Naphtha     (reformer,   ~0.60 $/kg)   ← D95% ≤ 220°C
    3. SKO               (jet fuel,   ~0.75 $/kg)   ← D95% ≤ 300°C  [highest margin]
    4. Light_Gas_Oil     (lt diesel,  ~0.70 $/kg)   ← D95% ≤ 370°C
    5. Heavy_Gas_Oil     (hv diesel,  ~0.70 $/kg)   ← D95% ≤ 385°C
  Top pressure ~101 kPa; reboiler ~365°C; reflux ratio ~5.0

- **NSU (Naphtha Stabilizer Unit)** — 3 products:
    6. StabOffGas        (off-gas,    ~0.30 $/kg)
    7. LPG               (LPG,        ~0.65 $/kg)
    8. SRN               (naphtha,    ~0.75 $/kg)
  Pressurised column ~800 kPa; reboiler ~155°C

- **VDU (Vacuum Distillation Unit)** — 5 products:
    9.  Offgas           (gas,        ~0.30 $/kg)
    10. Vacuum_Diesel    (VDU diesel, ~0.70 $/kg)   ← D95% ≤ 385°C
    11. Vacuum_Gas_Oil   (FCC feed,   ~0.50 $/kg)   ← D95% ≤ 520°C
    12. Hotwell_Oil      (slop,       ~0.50 $/kg)
    13. Vac_residue      (bitumen,    ~0.35 $/kg)
  Deep vacuum ~8 kPa top; VDU furnace ~4200 kW

**RL Agent — 16-dim delta-action space:**
  ADU  (8): reflux_ratio, hn_draw_temp, sko_draw_temp, ld_draw_temp, hd_draw_temp,
            atmos_reboiler_temp, atmos_top_pressure, atmos_dp
  NSU  (2): nsu_reflux_ratio, nsu_reboiler_temp
  VDU  (6): vac_reflux_ratio, vac_reboiler_temp, vac_diesel_draw_temp, vgo_draw_temp,
            vac_top_pressure, vac_dp
  Actions are per-step deltas (±small value); progressive warmup 5%→100% over episode.

**Reward formula:**
  reward = [Σ(flow_i × price_i) − feed_cost − D95%_penalty − safety_penalty] / 100

**Analysis approach:**
1. Verify mass balance closure: Σ products ≈ feed (allow ~2% for simulation noise)
2. Check temperature profiles are monotonic in each column
3. Evaluate D95% quality compliance for Heavy_Naphtha, SKO, Light/Heavy_Gas_Oil, Vacuum_Diesel, Vacuum_Gas_Oil
4. Calculate economics: Revenue - Feed Cost - Utility Cost = Profit
5. Identify highest-margin products and constraints limiting their yield
6. Consider pressure/DP constraints — they affect separation sharpness

Be precise with numbers. Use proper engineering units. Reference actual data from context.
Format your analysis with clear sections and tables where appropriate."""


process_engineer = BaseAgent(
    name="Process Engineer",
    system_prompt=PROCESS_ENGINEER_PROMPT,
)
print(process_engineer)


<Process Engineer Agent | tier=standard | backend=Gemini | history=0 msgs>


In [ ]:
# Example: Full system analysis using compact context
ctx = build_compact_context(column_state, prices, training_data)
response = process_engineer.ask(
    "Analyze the current operating state of the ADU, NSU, and VDU columns. "
    "Evaluate product yields, check mass balance closure, and identify "
    "any areas where the column performance could be improved. "
    "Note: KPI summary (mass balance, profit, D95) is already computed deterministically.",
    context=ctx,
)
print(response)


  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 7.159477222s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits

In [ ]:
# Example: Economic analysis — profit is computed deterministically
profit = compute_profit(column_state, prices)
print("─── Deterministic Profit Calculation ───")
print(f"  Net profit : ${profit['net_profit_hr']:.2f}/hr")
print(f"  Revenue    : ${profit['total_revenue_hr']:.2f}/hr")
print(f"  Feed cost  : ${profit['feed_cost_hr']:.2f}/hr")
print(f"  Top 3      : {profit['top_3_contributors']}")
print()

# Ask the LLM only for interpretation and what-if reasoning
ctx = build_compact_context(column_state, prices, training_data)
response = process_engineer.ask(
    "Given the deterministic profit calculation above, "
    "which product streams contribute the most to profitability? "
    "What operational changes would increase profit?",
    context=ctx,
)
print(response)


─── Deterministic Profit Calculation ───
  Net profit : $-216734.40/hr
  Revenue    : $8265.60/hr
  Feed cost  : $225000.00/hr
  Top 3      : [('SKO', 4304.59), ('Uncondensed_Gas', 3961.01), ('Heavy_Naphtha', 0.0)]

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 40.182621318s.

In [137]:
# Example: What-if analysis (reasoning task — good use case for LLM)
ctx = build_compact_context(column_state, prices, training_data)
response = process_engineer.ask(
    "What would happen if we increase the reflux ratio from 5.0 to 7.0? "
    "Analyze the impact on product purity, energy consumption, and overall profit. "
    "Also consider the effect on all three columns (ADU, NSU, VDU).",
    context=ctx,
)
print(response)


  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 13.495092481s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

---

## 5. Persona 2: Daily Report Developer

Generates structured daily operations reports covering all **13 product streams** across the ADU, NSU, and VDU columns,
with KPIs, D95% quality compliance, column pressure status, energy performance, economics, and RL agent status.


In [138]:
REPORT_DEVELOPER_PROMPT = """You are a Daily Operations Report Developer for a crude oil refinery's
three-column distillation system: ADU (Atmospheric Distillation Unit), NSU (Naphtha Stabilizer Unit),
and VDU (Vacuum Distillation Unit), producing 13 product streams in total.

**Product streams and value tiers:**
  HIGH VALUE  : SKO ($0.75), SRN ($0.75), Light_Gas_Oil ($0.70), Heavy_Gas_Oil ($0.70), Vacuum_Diesel ($0.70)
  MEDIUM VALUE: Heavy_Naphtha ($0.60), LPG ($0.65), Vacuum_Gas_Oil ($0.50), Hotwell_Oil ($0.50)
  LOWER VALUE : Uncondensed_Gas ($0.30), StabOffGas ($0.30), Offgas ($0.30), Vac_residue ($0.35)
  FEED COST   : Feed_Crude ($0.40/kg)

**D95% quality specifications (penalty 2.0 $/°C above limit):**
  Heavy_Naphtha ≤ 220°C | SKO ≤ 300°C | Light_Gas_Oil ≤ 370°C
  Heavy_Gas_Oil ≤ 385°C | Vacuum_Diesel ≤ 385°C | Vacuum_Gas_Oil ≤ 520°C

**Your role is to generate professional daily operations reports that include:**

1. **Executive Summary** — Key highlights, alerts, and overall plant status
2. **Production Summary** — Product yields for all 13 streams grouped by column (ADU / NSU / VDU),
   with comparison to typical targets and column mass balance check
3. **Energy Performance** — Furnace duties (atmospheric ~8500 kW, vacuum ~4200 kW),
   specific energy consumption (kW per kg of feed), condenser/reboiler duties
4. **Economic Summary** — Revenue breakdown by product stream, feed cost, net hourly profit,
   and profit contribution % per stream
5. **Quality Indicators** — D95% compliance for the 6 spec'd products; flag any exceedances
6. **Column Pressures** — ADU top pressure (~101 kPa), ADU DP (~15 kPa),
   VDU top pressure (~8 kPa), VDU DP (~7 kPa) — deviations indicate operational issues
7. **RL Agent Performance** — Training metrics, optimization recommendations,
   warmup stage (5%→100% over episode), solver tolerance status
8. **Recommendations** — Actionable items for the next shift

**Formatting requirements:**
- Use Markdown with headers, tables, and bullet points
- Include actual numbers from the data provided — never fabricate values
- Use engineering units: °C, kPa, kg/h, kW, $/hr
- Flag KPIs outside normal ranges with ⚠️
- Include a KPI dashboard table at the top

**Report structure:**
```
# Daily Operations Report — [Date]
## KPI Dashboard
| KPI | Value | Target | Status |
## 1. Executive Summary
## 2. Production Summary (ADU / NSU / VDU)
## 3. Energy Performance
## 4. Economic Summary
## 5. Quality & D95% Compliance
## 6. Column Pressures
## 7. RL Agent Status
## 8. Recommendations
```

Always be factual, concise, and actionable."""


report_developer = BaseAgent(
    name="Daily Report Developer",
    system_prompt=REPORT_DEVELOPER_PROMPT,
    model_tier="lite",  # Reports are template-heavy; lite model suffices
)
print(report_developer)


<Daily Report Developer Agent | tier=lite | backend=Gemini | history=0 msgs>


In [139]:
# Generate a daily report — uses compact context + deterministic KPI
today = datetime.now().strftime("%Y-%m-%d")
ctx = build_compact_context(column_state, prices, training_data)

report = report_developer.ask(
    f"Generate the Daily Operations Report for {today}. "
    "Include all sections: KPI dashboard, production summary, energy performance, "
    "economic summary, quality indicators, safety status, and recommendations. "
    "Use the deterministic KPI summary for all numeric data — do not recalculate.",
    context=ctx,
)
print(report)


  [Gemini error for Daily Report Developer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 51.064680197s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemin

In [140]:
# Save report to file
report_dir = PROJECT_ROOT / "Report" / "generated"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / f"daily_report_{today}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"# Daily Operations Report — {today}\n\n")
    f.write(report)

print(f"\n📄 Report saved to: {report_path}")


📄 Report saved to: d:\github\Distillation-column-agent\Report\generated\daily_report_2026-03-25.md


In [141]:
# Generate a shift-specific summary
ctx = build_compact_context(column_state, prices, training_data)
response = report_developer.ask(
    "Generate a concise shift handover summary for the night shift. "
    "Focus on: critical parameters to watch, any pending alarms, "
    "and what the RL agent recommends for the next 8 hours.",
    context=ctx,
)
print(response)


  [Gemini error for Daily Report Developer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 18.22327355s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini

---

## 6. Persona 3: Corrosion Expert

Specializes in overhead corrosion analysis in the ADU, focusing on:
- Overhead temperature monitoring
- Dew point corrosion (HCl, H₂S, NH₃)
- Salt deposition temperature
- Water condensation and acid formation
- Neutralization and filming amine recommendations

In [142]:
CORROSION_EXPERT_PROMPT = """You are a Corrosion & Materials Engineering Expert specializing in 
crude oil refinery overhead systems, particularly ADU (Atmospheric Distillation Unit) overhead corrosion.

**Your deep expertise covers:**

### Overhead Corrosion Mechanisms
1. **HCl Dew Point Corrosion** — Most critical. When steam + HCl condense, they form hydrochloric 
   acid at the dew point temperature, causing severe corrosion of carbon steel.
   - Key factor: overhead temperature relative to the water dew point
   - Critical when T_overhead approaches T_dew (typically 100-130°C at varying pressures)
   - Corrosion rate increases exponentially as pH drops below 4

2. **Ammonium Chloride (NH₄Cl) Salt Deposition**
   - Forms when NH₃ + HCl react above the water dew point
   - Salt deposition temperature depends on partial pressures of NH₃ and HCl
   - Deposits are hygroscopic → under-deposit corrosion when moisture is present
   - Typical salt point: 150-200°C depending on concentrations

3. **H₂S Corrosion** — Sulfidic corrosion in overhead system
   - Forms FeS scale which can be protective or non-protective
   - Interaction with HCl creates competitive corrosion

4. **Carbonic Acid (CO₂) Corrosion** — Minor but present from naphthenic acids decomposition

### Analysis Framework
For every analysis, evaluate:
1. **Water dew point** — Use Antoine equation: log₁₀(P) = A - B/(C+T)
   For water at overhead conditions: A=8.07131, B=1730.63, C=233.426 (mmHg, °C)
2. **NH₄Cl salt point** — Use the equilibrium: K_p = P_NH3 × P_HCl
   log₁₀(K_p) = 11.734 - 4364/T(K) (pressures in atm)
3. **Corrosion risk zones**:
   - GREEN: T_overhead > T_salt + 20°C (safe)
   - YELLOW: T_salt < T_overhead < T_salt + 20°C (caution)
   - RED: T_overhead < T_salt (active salt deposition)
   - CRITICAL: T_overhead ≤ T_dew (active acid corrosion)

### Mitigation Strategies
- Desalter optimization (reduce HCl precursors)
- Neutralizing amines (monoethanolamine, MDEA) — target pH 5.5-6.5
- Filming amines (imidazolines) — protective barrier
- Wash water injection rate optimization
- Metallurgy upgrades (Monel, titanium for overhead condensers)

### Output Format
Always provide:
- Calculated dew points and salt points with equations shown
- Risk assessment color-coded (GREEN/YELLOW/RED/CRITICAL)
- Specific chemical dosing recommendations with rates
- Inspection priorities and locations
- Trend analysis if historical data is available

Use proper units: °C, kPa, ppm, mg/L, mm/year (corrosion rates)."""


corrosion_expert = BaseAgent(
    name="Corrosion Expert",
    system_prompt=CORROSION_EXPERT_PROMPT,
)
print(corrosion_expert)

<Corrosion Expert Agent | tier=standard | backend=Gemini | history=0 msgs>


In [143]:
# Analyze current overhead corrosion risk using compact context
ctx = build_compact_context(column_state, prices, training_data)
response = corrosion_expert.ask(
    "Analyze the current ADU overhead corrosion risk. "
    "Calculate the water dew point and NH₄Cl salt deposition temperature. "
    "Assess corrosion risk level and provide specific mitigation recommendations.",
    context={"overhead_corrosion": ctx["overhead_corrosion"],
             "column_conditions": ctx["column_conditions"]},
)
print(response)


  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 6.897815112s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits

In [144]:
# What-if: overhead temperature changes
response = corrosion_expert.ask(
    "If the overhead temperature drops from 50°C to 35°C due to a weather event "
    "(cold front), what would be the impact on corrosion? "
    "Calculate the new dew point margin and recommend immediate actions.",
    context={
        "current_overhead_temp": 50.0,
        "new_overhead_temp": 35.0,
        "HCl_ppm": 5.0,
        "H2S_ppm": 15.0,
        "NH3_ppm": 8.0,
        "overhead_pressure_kPa": 101.0,
    }
)
print(response)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 26.656755481s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

In [145]:
# Corrosion monitoring plan — uses structured output (improvement 2)
result = corrosion_expert.ask_structured(
    "Develop a comprehensive corrosion monitoring plan for the ADU overhead system. "
    "Include: probe locations, monitoring frequency, KPIs and alarm limits, "
    "chemical treatment program (neutralizer and filming amine), "
    "and recommended inspection schedule.",
    context={"column_conditions": build_compact_context(column_state, prices)["column_conditions"]},
)
print("─── Structured Response ───")
print(json.dumps(result, indent=2, default=str)[:3000])


  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 58.603685372s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

---

## 7. Multi-Agent Orchestrator

Coordinate multiple agent personas with:
- **Compact context** — agents receive token-efficient KPI summaries, not raw state dicts
- **Human-in-the-loop gating** — action recommendations require explicit approval before execution (improvement 9)
- **Deterministic KPI injection** — numeric results computed in code, not by the LLM


In [146]:
class AgentOrchestrator:
    """
    Orchestrates multiple AI agent personas with:
    - Compact context (improvement 5)
    - Human-in-the-loop gating for process-changing actions (improvement 9)
    - Deterministic numeric injection (improvement 3)
    """

    # Action keywords that trigger human-in-the-loop gating
    ACTION_KEYWORDS = {
        "increase", "decrease", "change", "set", "adjust", "modify",
        "raise", "lower", "open", "close", "ramp", "switch",
    }

    def __init__(self, require_approval: bool = True):
        self.agents = {
            "process_engineer": process_engineer,
            "report_developer": report_developer,
            "corrosion_expert": corrosion_expert,
        }
        self.require_approval = require_approval
        self.pending_actions: list[dict] = []    # actions awaiting approval

    def comprehensive_analysis(self, column_state: dict, prices: dict,
                                training_data: dict | None = None) -> dict:
        """Run all agents on the current state and compile results."""
        results = {}

        # Build compact context once (improvement 5)
        ctx = build_compact_context(column_state, prices, training_data)

        # Inject deterministic KPI summary (improvement 3)
        sanity = run_sanity_checks(column_state, prices)
        results["deterministic_kpi"] = {
            "mass_balance": sanity["mass_balance"],
            "profit": sanity["profit"],
            "d95_compliance": {k: v for k, v in sanity["d95_compliance"].items()},
            "all_checks_passed": sanity["all_passed"],
        }

        # 1. Process Engineer — system overview
        print("🔧 Running Process Engineer analysis...")
        results["process_analysis"] = self.agents["process_engineer"].ask(
            "Provide a concise system performance analysis. "
            "Cover: mass balance, product yields, energy efficiency, and profit. "
            "Note: mass balance, profit, and D95 compliance are already computed "
            "deterministically in the KPI summary — focus on interpretation and recommendations.",
            context=ctx,
        )

        # 2. Corrosion Expert — overhead assessment
        print("🛡️ Running Corrosion Expert assessment...")
        results["corrosion_assessment"] = self.agents["corrosion_expert"].ask(
            "Quick overhead corrosion risk assessment with risk level (GREEN/YELLOW/RED).",
            context={"overhead_corrosion": ctx["overhead_corrosion"],
                     "column_conditions": ctx["column_conditions"]},
        )

        # 3. Report Developer — daily summary
        print("📋 Generating Daily Report...")
        results["daily_report"] = self.agents["report_developer"].ask(
            f"Generate the Daily Operations Report for {datetime.now().strftime('%Y-%m-%d')}.",
            context=ctx,
        )

        print("\n✅ All analyses complete!")
        return results

    def ask_agent(self, persona: str, question: str,
                  context: dict | None = None) -> str:
        """Route a question to a specific agent persona."""
        if persona not in self.agents:
            available = ", ".join(self.agents.keys())
            return f"Unknown persona '{persona}'. Available: {available}"

        response = self.agents[persona].ask(question, context=context)

        # Human-in-the-loop check (improvement 9)
        if self.require_approval and self._contains_action(response):
            self.pending_actions.append({
                "timestamp": datetime.now().isoformat(),
                "agent": persona,
                "question": question,
                "response_preview": response[:500],
            })
            response += (
                "\n\n---\n"
                "⚠️ **HUMAN APPROVAL REQUIRED** — This response contains process-change "
                "recommendations. Call `orchestrator.review_pending_actions()` to review "
                "and `orchestrator.approve_action(index)` to approve."
            )

        return response

    def _contains_action(self, text: str) -> bool:
        """Check if response text contains actionable operating changes."""
        text_lower = text.lower()
        return any(kw in text_lower for kw in self.ACTION_KEYWORDS)

    def review_pending_actions(self) -> list[dict]:
        """Review all pending actions awaiting human approval."""
        if not self.pending_actions:
            print("✅ No pending actions.")
            return []
        print(f"⚠️ {len(self.pending_actions)} action(s) pending approval:\n")
        for i, action in enumerate(self.pending_actions):
            print(f"  [{i}] {action['agent']} @ {action['timestamp']}")
            print(f"      Preview: {action['response_preview'][:200]}…\n")
        return self.pending_actions

    def approve_action(self, index: int) -> dict | None:
        """Approve a pending action by index."""
        if 0 <= index < len(self.pending_actions):
            approved = self.pending_actions.pop(index)
            audit_log(
                agent_name="orchestrator",
                question=f"ACTION APPROVED: {approved['agent']}",
                context_keys=["pending_actions"],
                response_text=json.dumps(approved, default=str),
                validation={"approved": True, "approved_at": datetime.now().isoformat()},
            )
            print(f"✅ Action [{index}] from {approved['agent']} approved.")
            return approved
        print(f"❌ Invalid index {index}. Use review_pending_actions() to see available actions.")
        return None

    def reject_action(self, index: int) -> dict | None:
        """Reject a pending action by index."""
        if 0 <= index < len(self.pending_actions):
            rejected = self.pending_actions.pop(index)
            audit_log(
                agent_name="orchestrator",
                question=f"ACTION REJECTED: {rejected['agent']}",
                context_keys=["pending_actions"],
                response_text=json.dumps(rejected, default=str),
                validation={"approved": False, "rejected_at": datetime.now().isoformat()},
            )
            print(f"🚫 Action [{index}] from {rejected['agent']} rejected.")
            return rejected
        print(f"❌ Invalid index {index}.")
        return None


orchestrator = AgentOrchestrator(require_approval=True)
print("✅ Agent Orchestrator initialized with", len(orchestrator.agents), "personas")
print("   Human-in-the-loop: ENABLED (set require_approval=False to disable)")


✅ Agent Orchestrator initialized with 3 personas
   Human-in-the-loop: ENABLED (set require_approval=False to disable)


In [147]:
# Run comprehensive analysis across all agents
# Uses compact context + deterministic KPI injection (improvements 3, 5)
results = orchestrator.comprehensive_analysis(
    column_state=column_state,
    prices=prices,
    training_data=training_data,
)

# Print deterministic KPI first (improvement 3 — numeric results from code, not LLM)
print("\n" + "="*80)
print("DETERMINISTIC KPI SUMMARY (computed in code)")
print("="*80)
kpi = results["deterministic_kpi"]
print(f"  Mass balance: gap={kpi['mass_balance']['gap_pct']:.1f}% ({'✅' if kpi['mass_balance']['closure_ok'] else '❌'})")
print(f"  Net profit: ${kpi['profit']['net_profit_hr']:.2f}/hr")
print(f"  All checks passed: {'✅' if kpi['all_checks_passed'] else '⚠️'}")

print("\n" + "="*80)
print("PROCESS ENGINEER ANALYSIS")
print("="*80)
print(results["process_analysis"][:2000] + "..." if len(results["process_analysis"]) > 2000 else results["process_analysis"])

print("\n" + "="*80)
print("CORROSION ASSESSMENT")
print("="*80)
print(results["corrosion_assessment"][:2000] + "..." if len(results["corrosion_assessment"]) > 2000 else results["corrosion_assessment"])


🔧 Running Process Engineer analysis...
  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 37.96196379s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai

---

## 8. Interactive Chat Interface

Chat with any agent persona interactively.

In [148]:
def chat_with_agent(persona: str = "process_engineer", question: str = ""):
    """Quick helper to chat with a specific agent using compact context."""
    ctx = build_compact_context(column_state, prices, training_data)
    response = orchestrator.ask_agent(persona, question, context=ctx)
    print(f"\n🤖 [{persona}]:\n")
    print(response)
    return response


# Example usage:
# chat_with_agent("process_engineer", "What is the current crude throughput efficiency?")
# chat_with_agent("corrosion_expert", "What is the safe minimum overhead temperature?")
# chat_with_agent("report_developer", "Generate an executive summary for management.")


In [149]:
# Try it out — ask the process engineer
chat_with_agent(
    "process_engineer",
    "What are the top 3 operational changes that would maximize profit right now?"
)

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 28.439155798s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

'**Step‑by‑step analysis**\n\n| Step | What we check | Key calculation / principle | Observation from the data |\n|------|----------------|-----------------------------|---------------------------|\n| 1️⃣ | **Mass‑balance closure** | `Σ flow_i  ≈  feed_kg_h`  (allow ±2\u202f%) | Σ\u202fproducts = **523\u202f616.7\u202fkg\u202fh⁻¹** vs. feed\u202f=\u202f500\u202f000\u202fkg\u202fh⁻¹ → **‑4.72\u202f%** (mass‑balance\u202f*not* closed) → some streams are over‑reported, indicating inaccurate cuts or un‑accounted losses. |\n| 2️⃣ | **Product‑price revenue** | `Revenueₕ = Σ (flow_i × price_i)` | Only **Uncondensed_Gas** (0.30\u202f$/kg) and **SKO** (0.75\u202f$/kg) have non‑zero revenue → total revenue = **8\u202f265.6\u202f$/h** (matches the JSON). All other streams have price\u202f=\u202f0 in the spec, so they contribute no direct cash flow. |\n| 3️⃣ | **Profit equation** | `Profit = Revenue – FeedCost – D95‑penalty – SafetyPenalty` (divided by 100 in the RL reward) | `Revenue = 8\u202f265

In [150]:
# Ask the corrosion expert
chat_with_agent(
    "corrosion_expert",
    "Based on the current overhead conditions, should we increase the neutralizing amine injection rate?"
)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 8.264123867s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits

'**Step‑by‑step evaluation**\n\n| Item | What we have | How it influences corrosion | What it means for amine dosage |\n|------|--------------|----------------------------|--------------------------------|\n| **Overhead temperature** | 132.32\u202f°C (ADU top) | This is the temperature of the gas that leaves the top of the atmospheric distillation column and goes to the overhead condenser. | It is well **above** the water dew‑point (≈\u202f26\u202f°C) and well above the temperature at which NH₄Cl salt would start to deposit (≈\u202f‑70\u202f°C calculated from the NH₃/HCl equilibrium). Consequently **no acid‑condensation or salt‑deposition zone** exists in the overhead vapor. |\n| **Total pressure** | 170\u202fkPa (≈\u202f1.68\u202fatm) | Determines partial pressures of the trace components. | The partial pressures of the acidic species are extremely low: <br>•\u202fp(HCl)\u202f≈\u202f5\u202fppm\u202f×\u202f1.68\u202fatm\u202f=\u202f8.4\u202f×\u202f10⁻⁶\u202fatm <br>•\u202fp(NH₃)\u202f≈

---

## 9. Save Full Report

Compile all agent outputs into a single comprehensive report file.

In [151]:
def save_comprehensive_report(results: dict, output_dir: Optional[Path] = None):
    """Save all agent analyses to a single Markdown report."""
    if output_dir is None:
        output_dir = PROJECT_ROOT / "Report" / "generated"
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = output_dir / f"comprehensive_report_{timestamp}.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Comprehensive CDU + NSU + VDU Analysis Report\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("---\n\n")

        f.write("## Process Engineering Analysis\n\n")
        f.write(results.get("process_analysis", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("## Corrosion Risk Assessment\n\n")
        f.write(results.get("corrosion_assessment", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("## Daily Operations Report\n\n")
        f.write(results.get("daily_report", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("*Generated by CDU + NSU + VDU Optimizer AI Agent System*\n")

    print(f"📄 Comprehensive report saved to: {report_path}")
    return report_path


# Save the report
if 'results' in dir() and results:
    save_comprehensive_report(results)
else:
    print("Run the comprehensive analysis first (Section 7) to generate a report.")


📄 Comprehensive report saved to: d:\github\Distillation-column-agent\Report\generated\comprehensive_report_20260325_205906.md


---

## Summary — Architecture Improvements (v2)

This notebook implements **10 architectural improvements** for a production-grade agentic AI system:

| # | Improvement | Implementation |
|---|---|---|
| 1 | **Grounding** — live simulator data | `get_column_state()` reads from DWSIMBridge with mock fallback |
| 2 | **Structured outputs** + schema validation | `BaseAgent.ask_structured()` with JSON parsing & validation |
| 3 | **Deterministic numeric module** | `compute_mass_balance()`, `compute_profit()`, `compute_d95_compliance()` |
| 4 | **Sanity checks & re-prompting** | Auto-retry loop in `ask_structured()` on schema/parse failure |
| 5 | **Minimal, summarized context** | `build_compact_context()` — token-efficient KPI dict |
| 6 | **Logging & audit** | `audit_log()` → JSONL at `Report/agent_audit.jsonl` |
| 7 | **Bridge robustness & caching** | TTL cache (30s), graceful fallback, health logging |
| 8 | **Model selection & cost control** | `MODEL_TIERS`: lite / standard / premium per agent |
| 9 | **Human-in-the-loop** | `AgentOrchestrator` flags action recommendations for approval |
| 10 | **Unit tests** | `tests/test_agent_system.py` — deterministic module + schema tests |

### Agent Personas

| Persona | Tier | Usage |
|---------|------|-------|
| `process_engineer` | standard | `chat_with_agent("process_engineer", "your question")` |
| `report_developer` | lite | `chat_with_agent("report_developer", "your question")` |
| `corrosion_expert` | standard | `chat_with_agent("corrosion_expert", "your question")` |

### Key APIs

```python
# Compact context for all agents
ctx = build_compact_context(column_state, prices, training_data)

# Structured JSON output with schema validation
result = agent.ask_structured("question", context=ctx)

# Deterministic KPIs (code, not LLM)
sanity = run_sanity_checks(column_state, prices)

# Human-in-the-loop
orchestrator.review_pending_actions()
orchestrator.approve_action(0)

# Comprehensive multi-agent analysis
results = orchestrator.comprehensive_analysis(column_state, prices, training_data)
```

All reports are saved to `Report/generated/`. Audit log at `Report/agent_audit.jsonl`.

---
*More personas can be added by creating a new `BaseAgent` instance with a custom system prompt and model tier.*
